In [7]:
#q1

import random

# First-Choice Hill Climbing
def first_choice_hc(landscape, start):
    current = start
    path = [current]
    n = len(landscape)

    while True:
        current_value = landscape[current - 1]
        moved = False

        # Check left neighbour first
        if current > 1:
            left = current - 1
            if landscape[left - 1] > current_value:
                current = left
                path.append(current)
                moved = True
                continue

        # Then check right neighbour
        if current < n:
            right = current + 1
            if landscape[right - 1] > current_value:
                current = right
                path.append(current)
                moved = True
                continue

        # No improving move found
        if not moved:
            break

    return path, current


# Stochastic Hill Climbing
def stochastic_hc(landscape, start):
    current = start
    path = [current]
    n = len(landscape)

    while True:
        current_value = landscape[current - 1]
        better_neighbors = []

        # Check left neighbour
        if current > 1:
            if landscape[current - 2] > current_value:
                better_neighbors.append(current - 1)

        # Check right neighbour
        if current < n:
            if landscape[current] > current_value:
                better_neighbors.append(current + 1)

        # If no better neighbours, stop
        if not better_neighbors:
            break

        # Randomly choose one
        current = random.choice(better_neighbors)
        path.append(current)

    return path, current


def run_experiment():
    landscape = [4, 9, 6, 11, 8, 15, 10, 7, 13, 5, 16, 12]

    trials = 50
    reach_11 = 0
    local_max = 0

    for _ in range(trials):
        path, terminal = stochastic_hc(landscape, 4)

        if terminal == 11:
            reach_11 += 1
        else:
            local_max += 1

    print("\n--- Stochastic HC Experiment (start = 4) ---")
    print(f"Reached state 11: {reach_11} times")
    print(f"Reached local maximum: {local_max} times")
    print(f"Success rate: {(reach_11 / trials) * 100:.2f}%")


# Main function
def main():
    landscape = [4, 9, 6, 11, 8, 15, 10, 7, 13, 5, 16, 12]

    print(f"{'Start':<6}{'Algorithm':<15}{'Path':<30}{'Terminal':<10}{'Steps'}")
    print("-" * 80)

    for s in range(1, 13):
        # First-Choice HC
        path, terminal = first_choice_hc(landscape, s)
        print(f"{s:<6}{'First-Choice':<15}{str(path):<30}{terminal:<10}{len(path)-1}")

        # Stochastic HC
        path, terminal = stochastic_hc(landscape, s)
        print(f"{s:<6}{'Stochastic':<15}{str(path):<30}{terminal:<10}{len(path)-1}")

    # ✅ Part (b) experiment: 50 runs from s = 4
    trials = 50
    reach_11 = 0
    local_max = 0

    for _ in range(trials):
        path, terminal = stochastic_hc(landscape, 4)

        if terminal == 11:
            reach_11 += 1
        else:
            local_max += 1

    print("\n--- Stochastic HC Experiment (start = 4) ---")
    print(f"Reached state 11: {reach_11} times")
    print(f"Reached local maximum: {local_max} times")
    print(f"Success rate: {(reach_11 / trials) * 100:.2f}%")


if __name__ == "__main__":
    main()

Start Algorithm      Path                          Terminal  Steps
--------------------------------------------------------------------------------
1     First-Choice   [1, 2]                        2         1
1     Stochastic     [1, 2]                        2         1
2     First-Choice   [2]                           2         0
2     Stochastic     [2]                           2         0
3     First-Choice   [3, 2]                        2         1
3     Stochastic     [3, 4]                        4         1
4     First-Choice   [4]                           4         0
4     Stochastic     [4]                           4         0
5     First-Choice   [5, 4]                        4         1
5     Stochastic     [5, 4]                        4         1
6     First-Choice   [6]                           6         0
6     Stochastic     [6]                           6         0
7     First-Choice   [7, 6]                        6         1
7     Stochastic     [7, 6]      

In [8]:


# First-Choice Hill Climbing (with plateau detection + sideways moves)
def first_choice_hc(landscape, start, sideways_limit=0):
    current = start
    path = [current]
    n = len(landscape)
    sideways_moves = 0

    while True:
        current_value = landscape[current - 1]
        moved = False
        equal_found = False

        # LEFT
        if current > 1:
            left = current - 1
            if landscape[left - 1] > current_value:
                current = left
                path.append(current)
                sideways_moves = 0
                moved = True
                continue
            elif landscape[left - 1] == current_value:
                equal_found = True
                if sideways_moves < sideways_limit:
                    print(f"Plateau at state {current}")
                    current = left
                    path.append(current)
                    sideways_moves += 1
                    moved = True
                    continue

        # RIGHT
        if current < n:
            right = current + 1
            if landscape[right - 1] > current_value:
                current = right
                path.append(current)
                sideways_moves = 0
                moved = True
                continue
            elif landscape[right - 1] == current_value:
                equal_found = True
                if sideways_moves < sideways_limit:
                    print(f"Plateau at state {current}")
                    current = right
                    path.append(current)
                    sideways_moves += 1
                    moved = True
                    continue

        # Plateau detection (no better move but equal exists)
        if not moved and equal_found and sideways_limit == 0:
            print(f"Plateau detected at state {current}")

        break

    return path, current


# Stochastic Hill Climbing (with plateau detection + sideways moves)
def stochastic_hc(landscape, start, sideways_limit=0):
    current = start
    path = [current]
    n = len(landscape)
    sideways_moves = 0

    while True:
        current_value = landscape[current - 1]
        better = []
        equal = []

        # LEFT
        if current > 1:
            val = landscape[current - 2]
            if val > current_value:
                better.append(current - 1)
            elif val == current_value:
                equal.append(current - 1)

        # RIGHT
        if current < n:
            val = landscape[current]
            if val > current_value:
                better.append(current + 1)
            elif val == current_value:
                equal.append(current + 1)

        if better:
            current = random.choice(better)
            path.append(current)
            sideways_moves = 0

        elif equal and sideways_moves < sideways_limit:
            print(f"Plateau at state {current}")
            current = random.choice(equal)
            path.append(current)
            sideways_moves += 1

        else:
            if equal and sideways_limit == 0:
                print(f"Plateau detected at state {current}")
            break

    return path, current


def main():
    # Modified landscape with plateau at states 5,6,7
    landscape = [4, 9, 6, 11, 15, 15, 15, 7, 13, 5, 16, 12]

    print("=== WITHOUT SIDEWAYS MOVES ===")
    stuck_plateau = 0

    for s in range(1, 13):
        _, t1 = first_choice_hc(landscape, s)
        _, t2 = stochastic_hc(landscape, s)

        if t1 in [5, 6, 7]:
            stuck_plateau += 1
        if t2 in [5, 6, 7]:
            stuck_plateau += 1

    print(f"\nRuns stuck on plateau: {stuck_plateau}")

    print("\n=== WITH SIDEWAYS MOVES (limit = 10) ===")
    success = 0

    for s in range(1, 13):
        _, t1 = first_choice_hc(landscape, s, sideways_limit=10)
        _, t2 = stochastic_hc(landscape, s, sideways_limit=10)

        if t1 == 11:
            success += 1
        if t2 == 11:
            success += 1

    print(f"\nTotal runs reaching state 11: {success}")


if __name__ == "__main__":
    main()

=== WITHOUT SIDEWAYS MOVES ===
Plateau detected at state 5
Plateau detected at state 5
Plateau detected at state 5
Plateau detected at state 5
Plateau detected at state 6
Plateau detected at state 6
Plateau detected at state 7
Plateau detected at state 7
Plateau detected at state 7

Runs stuck on plateau: 9

=== WITH SIDEWAYS MOVES (limit = 10) ===
Plateau at state 5
Plateau at state 6
Plateau at state 5
Plateau at state 6
Plateau at state 5
Plateau at state 6
Plateau at state 5
Plateau at state 6
Plateau at state 5
Plateau at state 6
Plateau at state 5
Plateau at state 6
Plateau at state 5
Plateau at state 6
Plateau at state 5
Plateau at state 6
Plateau at state 7
Plateau at state 6
Plateau at state 5
Plateau at state 6
Plateau at state 5
Plateau at state 6
Plateau at state 5
Plateau at state 6
Plateau at state 5
Plateau at state 6
Plateau at state 5
Plateau at state 6
Plateau at state 5
Plateau at state 6
Plateau at state 5
Plateau at state 6
Plateau at state 7
Plateau at state 6
Pla

In [10]:
import random

# ---------- Q1 FUNCTIONS ----------

def first_choice_hc(landscape, start):
    current = start
    path = [current]
    n = len(landscape)

    while True:
        current_value = landscape[current - 1]

        if current > 1 and landscape[current - 2] > current_value:
            current -= 1
            path.append(current)
            continue

        if current < n and landscape[current] > current_value:
            current += 1
            path.append(current)
            continue

        break

    return path, current


def stochastic_hc(landscape, start):
    current = start
    path = [current]
    n = len(landscape)

    while True:
        current_value = landscape[current - 1]
        neighbors = []

        if current > 1 and landscape[current - 2] > current_value:
            neighbors.append(current - 1)

        if current < n and landscape[current] > current_value:
            neighbors.append(current + 1)

        if not neighbors:
            break

        current = random.choice(neighbors)
        path.append(current)

    return path, current


# ---------- RANDOM RESTART HC ----------

def random_restart_hc(landscape, num_restarts, variant='first choice'):
    results = []
    best_state = None
    best_value = float('-inf')

    for _ in range(num_restarts):
        start = random.randint(1, len(landscape))  # random start

        if variant == 'first choice':
            path, terminal = first_choice_hc(landscape, start)
        else:
            path, terminal = stochastic_hc(landscape, start)

        value = landscape[terminal - 1]
        results.append((start, terminal, path))

        if value > best_value:
            best_value = value
            best_state = terminal

    return best_state, best_value, results


# ---------- FIND LOCAL MAXIMA ----------

def find_local_maxima(landscape):
    maxima = []
    n = len(landscape)

    for i in range(n):
        left = landscape[i - 1] if i > 0 else float('-inf')
        right = landscape[i + 1] if i < n - 1 else float('-inf')

        if landscape[i] > left and landscape[i] > right:
            maxima.append(i + 1)

    return maxima


def compute_p(landscape):
    success_states = []
    n = len(landscape)

    for start in range(1, n + 1):
        _, terminal = first_choice_hc(landscape, start)
        if terminal == 11:
            success_states.append(start)

    p = len(success_states) / n
    return p, success_states


# ---------- MAIN ----------

def main():
    landscape = [5, 8, 6, 12, 9, 7, 17, 14, 10, 6, 19, 15, 11, 8]
    num_restarts = 20
    global_max = max(landscape)

    # Print local maxima
    print("Local Maxima:", find_local_maxima(landscape))

    # Run both variants
    for variant in ['first choice', 'stochastic']:
        print(f"\n=== Random Restart HC ({variant}) ===")
        print(f"{'Restart':<8}{'Start':<8}{'Terminal':<10}{'f-value':<10}{'Global Max?'}")
        print("-" * 60)

        best_state, best_value, results = random_restart_hc(
            landscape, num_restarts, variant
        )

        for i, (start, terminal, path) in enumerate(results, 1):
            value = landscape[terminal - 1]
            is_global = "Yes" if value == global_max else "No"

            print(f"{i:<8}{start:<8}{terminal:<10}{value:<10}{is_global}")

        print(f"\nBest State Found: {best_state} (Value = {best_value})")


if __name__ == "__main__":
    main()

Local Maxima: [2, 4, 7, 11]

=== Random Restart HC (first choice) ===
Restart Start   Terminal  f-value   Global Max?
------------------------------------------------------------
1       10      7         17        No
2       7       7         17        No
3       8       7         17        No
4       10      7         17        No
5       11      11        19        Yes
6       4       4         12        No
7       8       7         17        No
8       1       2         8         No
9       10      7         17        No
10      2       2         8         No
11      3       2         8         No
12      2       2         8         No
13      13      11        19        Yes
14      11      11        19        Yes
15      7       7         17        No
16      5       4         12        No
17      14      11        19        Yes
18      8       7         17        No
19      4       4         12        No
20      5       4         12        No

Best State Found: 11 (Value = 19)

=

In [11]:
import random

#b

def first_choice_hc(landscape, start):
    current = start
    path = [current]
    n = len(landscape)

    while True:
        current_value = landscape[current - 1]

        if current > 1 and landscape[current - 2] > current_value:
            current -= 1
            path.append(current)
            continue

        if current < n and landscape[current] > current_value:
            current += 1
            path.append(current)
            continue

        break

    return path, current


def stochastic_hc(landscape, start):
    current = start
    path = [current]
    n = len(landscape)

    while True:
        current_value = landscape[current - 1]
        neighbors = []

        if current > 1 and landscape[current - 2] > current_value:
            neighbors.append(current - 1)

        if current < n and landscape[current] > current_value:
            neighbors.append(current + 1)

        if not neighbors:
            break

        current = random.choice(neighbors)
        path.append(current)

    return path, current


# ---------- RANDOM RESTART HC ----------

def random_restart_hc(landscape, num_restarts, variant='first choice'):
    results = []
    best_state = None
    best_value = float('-inf')

    for _ in range(num_restarts):
        start = random.randint(1, len(landscape))  # random start

        if variant == 'first choice':
            path, terminal = first_choice_hc(landscape, start)
        else:
            path, terminal = stochastic_hc(landscape, start)

        value = landscape[terminal - 1]
        results.append((start, terminal, path))

        if value > best_value:
            best_value = value
            best_state = terminal

    return best_state, best_value, results


# ---------- FIND LOCAL MAXIMA ----------

def find_local_maxima(landscape):
    maxima = []
    n = len(landscape)

    for i in range(n):
        left = landscape[i - 1] if i > 0 else float('-inf')
        right = landscape[i + 1] if i < n - 1 else float('-inf')

        if landscape[i] > left and landscape[i] > right:
            maxima.append(i + 1)

    return maxima


# ---------- PART B HELPERS ----------

def compute_p(landscape):
    success_states = []
    n = len(landscape)

    for start in range(1, n + 1):
        _, terminal = first_choice_hc(landscape, start)
        if terminal == 11:
            success_states.append(start)

    p = len(success_states) / n
    return p, success_states


def empirical_experiment(landscape, restart_values, trials=100):
    results = {}

    for n in restart_values:
        success_count = 0

        for _ in range(trials):
            best_state, _, _ = random_restart_hc(landscape, n, 'first choice')

            if best_state == 11:
                success_count += 1

        results[n] = success_count / trials

    return results


def theoretical_probs(p, restart_values):
    probs = {}

    for n in restart_values:
        probs[n] = 1 - (1 - p) ** n

    return probs


def compare_results(empirical, theoretical):
    print("\n=== Part (b) Results ===")
    print(f"{'n':<5}{'Empirical':<15}{'Theoretical':<15}")
    print("-" * 35)

    for n in empirical:
        print(f"{n:<5}{empirical[n]:<15.4f}{theoretical[n]:<15.4f}")


# ---------- MAIN ----------

def main():
    landscape = [5, 8, 6, 12, 9, 7, 17, 14, 10, 6, 19, 15, 11, 8]
    num_restarts = 20
    global_max = max(landscape)

    # Print local maxima
    print("Local Maxima:", find_local_maxima(landscape))

    # -------- PART A OUTPUT --------
    for variant in ['first choice', 'stochastic']:
        print(f"\n=== Random Restart HC ({variant}) ===")
        print(f"{'Restart':<8}{'Start':<8}{'Terminal':<10}{'f-value':<10}{'Global Max?'}")
        print("-" * 60)

        best_state, best_value, results = random_restart_hc(
            landscape, num_restarts, variant
        )

        for i, (start, terminal, path) in enumerate(results, 1):
            value = landscape[terminal - 1]
            is_global = "Yes" if value == global_max else "No"

            print(f"{i:<8}{start:<8}{terminal:<10}{value:<10}{is_global}")

        print(f"\nBest State Found: {best_state} (Value = {best_value})")

    # -------- PART B --------

    restart_values = [1, 3, 5, 10, 20]

    # Compute p
    p, success_states = compute_p(landscape)
    print("\nStates reaching global max (11):", success_states)
    print(f"p = {len(success_states)}/{len(landscape)} = {p:.4f}")

    # Empirical
    empirical = empirical_experiment(landscape, restart_values)

    # Theoretical
    theoretical = theoretical_probs(p, restart_values)

    # Compare
    compare_results(empirical, theoretical)


if __name__ == "__main__":
    main()

Local Maxima: [2, 4, 7, 11]

=== Random Restart HC (first choice) ===
Restart Start   Terminal  f-value   Global Max?
------------------------------------------------------------
1       12      11        19        Yes
2       4       4         12        No
3       4       4         12        No
4       12      11        19        Yes
5       1       2         8         No
6       7       7         17        No
7       2       2         8         No
8       1       2         8         No
9       2       2         8         No
10      6       4         12        No
11      1       2         8         No
12      4       4         12        No
13      4       4         12        No
14      2       2         8         No
15      6       4         12        No
16      14      11        19        Yes
17      6       4         12        No
18      12      11        19        Yes
19      5       4         12        No
20      10      7         17        No

Best State Found: 11 (Value = 19)

=

In [12]:
import random

# ---------- Q1 FUNCTIONS ----------

def first_choice_hc(landscape, start):
    current = start
    path = [current]
    n = len(landscape)

    while True:
        current_value = landscape[current - 1]

        if current > 1 and landscape[current - 2] > current_value:
            current -= 1
            path.append(current)
            continue

        if current < n and landscape[current] > current_value:
            current += 1
            path.append(current)
            continue

        break

    return path, current


def stochastic_hc(landscape, start):
    current = start
    path = [current]
    n = len(landscape)

    while True:
        current_value = landscape[current - 1]
        neighbors = []

        if current > 1 and landscape[current - 2] > current_value:
            neighbors.append(current - 1)

        if current < n and landscape[current] > current_value:
            neighbors.append(current + 1)

        if not neighbors:
            break

        current = random.choice(neighbors)
        path.append(current)

    return path, current


# ---------- RANDOM RESTART HC ----------

def random_restart_hc(landscape, num_restarts, variant='first choice'):
    results = []
    best_state = None
    best_value = float('-inf')
    plateau_count = 0   # (c) counts how many restarts end on plateau states (7 or 8)

    for _ in range(num_restarts):
        start = random.randint(1, len(landscape))

        if variant == 'first choice':
            path, terminal = first_choice_hc(landscape, start)
        else:
            path, terminal = stochastic_hc(landscape, start)

        value = landscape[terminal - 1]
        results.append((start, terminal, path))

        if terminal in [7, 8]:   # (c) check if search got stuck on plateau
            plateau_count += 1

        if value > best_value:
            best_value = value
            best_state = terminal

    return best_state, best_value, results, plateau_count


# ---------- FIND LOCAL MAXIMA ----------

def find_local_maxima(landscape):
    maxima = []
    n = len(landscape)

    for i in range(n):
        left = landscape[i - 1] if i > 0 else float('-inf')
        right = landscape[i + 1] if i < n - 1 else float('-inf')

        if landscape[i] > left and landscape[i] > right:
            maxima.append(i + 1)

    return maxima


# ---------- PART (b) ----------

def compute_p(landscape):
    success_states = []
    n = len(landscape)

    for start in range(1, n + 1):
        _, terminal = first_choice_hc(landscape, start)
        if terminal == 11:
            success_states.append(start)

    p = len(success_states) / n   # (b) probability that a single random start reaches global max
    return p, success_states


def empirical_experiment(landscape, restart_values, trials=100):
    results = {}

    for n in restart_values:
        success_count = 0

        for _ in range(trials):
            best_state, _, _, _ = random_restart_hc(landscape, n, 'first choice')

            if best_state == 11:
                success_count += 1

        results[n] = success_count / trials   # (b) empirical probability over 100 trials

    return results


def theoretical_probs(p, restart_values):
    probs = {}

    for n in restart_values:
        probs[n] = 1 - (1 - p) ** n   # (b) theoretical RRHC success probability

    return probs


def compare_results(empirical, theoretical):
    print("\n=== Part (b) Results ===")
    print(f"{'n':<5}{'Empirical':<15}{'Theoretical':<15}")
    print("-" * 35)

    for n in empirical:
        print(f"{n:<5}{empirical[n]:<15.4f}{theoretical[n]:<15.4f}")


# ---------- PART (c) ----------

def plateau_experiment(landscape, trials=100, restarts=20):
    success = 0
    total_plateau = 0

    for _ in range(trials):
        best_state, _, _, plateau_count = random_restart_hc(
            landscape, restarts, 'first choice'
        )

        if best_state == 11:
            success += 1

        total_plateau += plateau_count   # (c) accumulate plateau terminations

    discovery_rate = success / trials   # (c) probability of finding global max
    avg_plateau = total_plateau / trials   # (c) average plateau hits per run

    return discovery_rate, avg_plateau


# ---------- MAIN ----------

def main():
    landscape = [5, 8, 6, 12, 9, 7, 17, 14, 10, 6, 19, 15, 11, 8]
    num_restarts = 20
    global_max = max(landscape)

    print("Local Maxima:", find_local_maxima(landscape))

    for variant in ['first choice', 'stochastic']:
        print(f"\n=== Random Restart HC ({variant}) ===")
        print(f"{'Restart':<8}{'Start':<8}{'Terminal':<10}{'f-value':<10}{'Global Max?'}")
        print("-" * 60)

        best_state, best_value, results, _ = random_restart_hc(
            landscape, num_restarts, variant
        )

        for i, (start, terminal, path) in enumerate(results, 1):
            value = landscape[terminal - 1]
            is_global = "Yes" if value == global_max else "No"

            print(f"{i:<8}{start:<8}{terminal:<10}{value:<10}{is_global}")

        print(f"\nBest State Found: {best_state} (Value = {best_value})")

    # ----- PART (b) -----
    restart_values = [1, 3, 5, 10, 20]

    p, success_states = compute_p(landscape)
    print("\nStates reaching global max (11):", success_states)
    print(f"p = {len(success_states)}/{len(landscape)} = {p:.4f}")

    empirical = empirical_experiment(landscape, restart_values)
    theoretical = theoretical_probs(p, restart_values)

    compare_results(empirical, theoretical)

    # ----- PART (c) -----
    print("\n=== Part (c): Plateau Experiment ===")

    plateau_landscape = [5, 8, 6, 12, 9, 7, 17, 17, 10, 6, 19, 15, 11, 8]  # (c) plateau at states 7 and 8

    orig_rate, orig_plateau = plateau_experiment(landscape)
    plat_rate, plat_plateau = plateau_experiment(plateau_landscape)

    print(f"\n{'Case':<20}{'Discovery Rate':<20}{'Avg Plateau Hits'}")
    print("-" * 55)
    print(f"{'Original':<20}{orig_rate:<20.4f}{orig_plateau:.2f}")
    print(f"{'With Plateau':<20}{plat_rate:<20.4f}{plat_plateau:.2f}")


if __name__ == "__main__":
    main()

Local Maxima: [2, 4, 7, 11]

=== Random Restart HC (first choice) ===
Restart Start   Terminal  f-value   Global Max?
------------------------------------------------------------
1       2       2         8         No
2       9       7         17        No
3       3       2         8         No
4       8       7         17        No
5       6       4         12        No
6       10      7         17        No
7       12      11        19        Yes
8       13      11        19        Yes
9       12      11        19        Yes
10      4       4         12        No
11      8       7         17        No
12      13      11        19        Yes
13      3       2         8         No
14      7       7         17        No
15      9       7         17        No
16      12      11        19        Yes
17      4       4         12        No
18      5       4         12        No
19      11      11        19        Yes
20      12      11        19        Yes

Best State Found: 11 (Value = 19)

In [14]:
import random

# ===================== Q3(a): Diagnose HC =====================

def diagnose_hc(landscape, start):
    current = start
    visited = set()

    while True:
        if current in visited:
            print(f"Terminated at state {current} with f={landscape[current-1]}. Failure mode: Ridge")
            return
        visited.add(current)

        current_value = landscape[current - 1]

        neighbors = []
        if current > 1:
            neighbors.append(current - 1)
        if current < len(landscape):
            neighbors.append(current + 1)

        better = []
        equal = []

        for n in neighbors:
            if landscape[n - 1] > current_value:
                better.append(n)
            elif landscape[n - 1] == current_value:
                equal.append(n)

        if better:
            current = better[0]
        elif equal:
            print(f"Terminated at state {current} with f={current_value}. Failure mode: Plateau")
            return
        else:
            print(f"Terminated at state {current} with f={current_value}. Failure mode: Local Maximum")
            return


# Test landscapes
land_local = [1, 3, 7, 5, 2]
land_plateau = [1, 3, 5, 5, 4]
land_ridge = [1, 3, 2, 3, 1]


# ===================== Q3(b): N-Queens =====================

def count_conflicts(board):
    conflicts = 0
    n = len(board)

    for i in range(n):
        for j in range(i + 1, n):
            if board[i] == board[j] or abs(board[i] - board[j]) == abs(i - j):
                conflicts += 1

    return conflicts


def stochastic_hc_nqueens(board):
    current = board[:]
    current_conflicts = count_conflicts(current)

    while True:
        improved = False

        for _ in range(50):  # try multiple random swaps
            i, j = random.sample(range(len(board)), 2)

            new_board = current[:]
            new_board[i], new_board[j] = new_board[j], new_board[i]

            new_conflicts = count_conflicts(new_board)

            if new_conflicts < current_conflicts:
                current = new_board
                current_conflicts = new_conflicts
                improved = True
                break

        if not improved:
            break

    return current, current_conflicts


def solve_nqueens_rrhc(num_restarts):
    for restart in range(1, num_restarts + 1):
        board = list(range(8))   # better initialization
        random.shuffle(board)

        solution, conflicts = stochastic_hc_nqueens(board)

        if conflicts == 0:
            return restart, solution

    return None, None


def print_board(board):
    for r in range(8):
        row = ""
        for c in range(8):
            row += "Q " if board[c] == r else ". "
        print(row)


# ===================== Q3(c): Benchmark =====================

def benchmark():
    ks = [5, 10, 25, 50, 100]
    trials = 30

    print("\n=== Benchmark Results ===")
    print(f"{'k':<10}{'Success Rate':<20}{'Avg Restarts'}")
    print("-" * 45)

    for k in ks:
        success = 0
        total_restarts = 0

        for _ in range(trials):
            restarts, solution = solve_nqueens_rrhc(k)

            if solution:
                success += 1
                total_restarts += restarts

        success_rate = success / trials
        avg_restarts = total_restarts / success if success > 0 else 0

        print(f"{k:<10}{success_rate:<20.2f}{avg_restarts:.2f}")


# ===================== MAIN =====================

def main():
    print("=== Q3(a): Diagnosis ===")
    diagnose_hc(land_local, 3)
    diagnose_hc(land_plateau, 3)
    diagnose_hc(land_ridge, 2)

    print("\n=== Q3(b): N-Queens ===")
    restarts, solution = solve_nqueens_rrhc(100)

    if solution:
        print(f"Solution found in {restarts} restarts")
        print("Board:", solution)
        print_board(solution)
    else:
        print("No solution found")

    print("\n=== Q3(c): Benchmark ===")
    benchmark()


if __name__ == "__main__":
    main()

=== Q3(a): Diagnosis ===
Terminated at state 3 with f=7. Failure mode: Local Maximum
Terminated at state 3 with f=5. Failure mode: Plateau
Terminated at state 2 with f=3. Failure mode: Local Maximum

=== Q3(b): N-Queens ===
Solution found in 2 restarts
Board: [0, 6, 3, 5, 7, 1, 4, 2]
Q . . . . . . . 
. . . . . Q . . 
. . . . . . . Q 
. . Q . . . . . 
. . . . . . Q . 
. . . Q . . . . 
. Q . . . . . . 
. . . . Q . . . 

=== Q3(c): Benchmark ===

=== Benchmark Results ===
k         Success Rate        Avg Restarts
---------------------------------------------
5         0.87                2.58
10        0.97                2.83
25        1.00                3.07
50        1.00                2.97
100       1.00                2.70
